# Uncertainty-Flagged Visual-Textual Explanations for Diabetic Retinopathy Grading on APTOS 2019

**Study.** A fundus-image classifier grades diabetic retinopathy (0–4). Each prediction comes with
(i) an **uncertainty flag** (Monte Carlo dropout) that refers doubtful cases to a clinician,
(ii) a **visual explanation** (Grad-CAM heatmap) and (iii) a short **textual explanation** generated from a template.
The study tests whether these can be trusted:
- **RQ1** — Is the uncertainty flag a reliable safety gate? (errors caught and quality of the retained cases at 10/20/30 % referral)
- **RQ2** — Are visual explanations faithful and stable, and are they worse for flagged cases?
- **RQ3** — Are textual explanations faithful: does hiding the region the text names reduce confidence more than hiding a random region?

**Data.** APTOS 2019 Blindness Detection (3,662 graded fundus images, Kaggle).
Default source: a public Hugging Face copy resized to 224×224 (`bumbledeep/aptos`, no login).
Alternative: the original Kaggle files (needs a Kaggle API token and accepting the competition rules) — set `DATA_SOURCE = 'kaggle'`.

### How to run
1. **Runtime → Change runtime type → T4 GPU.** Save.
2. Leave `PILOT = True` in Step 2 and **Runtime → Run all** (about 10 minutes). Allow Google Drive access.
3. Check the pilot output, set `PILOT = False`, and **Run all** again (about 45–90 minutes).
4. If Colab disconnects, reconnect and **Run all**: finished seeds are read from Drive and skipped.

## Step 1 — Install libraries

In [ ]:
!pip -q install imagehash datasets

## Step 2 — Settings and Google Drive (edit only this cell)

In [ ]:
import os, time, json, platform, warnings, glob, io, math, textwrap
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

PILOT = True              # True = quick test; False = full study
DATA_SOURCE = 'hf'        # 'hf' (Hugging Face copy, no login) or 'kaggle' (original files; needs kaggle.json)
SYNTHETIC_TEST = False    # leave False (only used to self-test the code without internet)

IMG = 224
GRADES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
DROPOUT_P = 0.3
MC_SAMPLES = 30
REFERRAL_RATES = [0.0, 0.1, 0.2, 0.3]
MAIN_REFERRAL = 0.2        # operating point used for the flagged / unflagged comparison
MASK_FRACTION = 0.2        # share of the image hidden in the faithfulness test
DUP_HAMMING = 4            # perceptual-hash distance treated as a near-duplicate

if PILOT:
    SEEDS, EPOCHS, MAX_IMAGES, RUN_TAG = [0], 2, 600, 'pilot'
else:
    SEEDS, EPOCHS, MAX_IMAGES, RUN_TAG = [0, 1, 2], 15, None, 'full'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/aptos_xai_study'
except ImportError:
    OUT_DIR = os.path.abspath('aptos_xai_study')
os.makedirs(OUT_DIR, exist_ok=True)
print('Mode:', RUN_TAG, '| seeds:', SEEDS, '| epochs:', EPOCHS, '| source:', DATA_SOURCE)

## Step 3 — Load APTOS 2019 (224 × 224) and find near-duplicate images
Images are cached in Google Drive as one compressed `.npz` file. Near-duplicates (perceptual hash) are grouped so that
copies of the same eye image can never fall into both the training and the test set.

In [ ]:
from PIL import Image
import imagehash
CACHE = os.path.join(OUT_DIR, 'aptos224.npz')

def _synthetic(n=900, seed=0):
    rng = np.random.default_rng(seed); X = np.zeros((n, IMG, IMG, 3), np.uint8)
    y = rng.choice(5, n, p=[.45, .1, .25, .08, .12]); yy, xx = np.mgrid[:IMG, :IMG]
    disk = ((yy - IMG / 2) ** 2 + (xx - IMG / 2) ** 2) < (IMG * 0.45) ** 2
    for i in range(n):
        im = np.zeros((IMG, IMG, 3)); im[disk] = [150, 70, 30]
        for _ in range(int(y[i]) * 6):   # "lesions": number grows with grade
            cy, cx = rng.integers(40, IMG - 40, 2); r = rng.integers(3, 8)
            im[((yy - cy) ** 2 + (xx - cx) ** 2) < r ** 2] = [230, 220, 120]
        X[i] = np.clip(im + rng.normal(0, 12, im.shape), 0, 255)
    X[n - 20:] = X[:20]; y[n - 20:] = y[:20]   # plant duplicates to test the grouping
    return X, y

def _to224(img):
    img = img.convert('RGB')
    return np.asarray(img.resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)

def load_aptos():
    if os.path.exists(CACHE):
        d = np.load(CACHE); return d['X'], d['y']
    if SYNTHETIC_TEST:
        X, y = _synthetic()
    elif DATA_SOURCE == 'hf':
        from datasets import load_dataset
        ds = load_dataset('bumbledeep/aptos', split='train')
        lab_col = next(c for c in ['label_code', 'diagnosis', 'label'] if c in ds.column_names)
        img_col = next(c for c in ['image', 'img'] if c in ds.column_names)
        X = np.stack([_to224(r[img_col]) for r in ds]); y = np.asarray(ds[lab_col]).astype(int)
    else:   # original Kaggle files: upload kaggle.json first (see instructions)
        os.makedirs('/root/.kaggle', exist_ok=True)
        if os.path.exists('/content/kaggle.json'):
            os.system('cp /content/kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json')
        os.system('pip -q install kaggle && kaggle competitions download -c aptos2019-blindness-detection -f train.csv -p /content/aptos')
        os.system('kaggle competitions download -c aptos2019-blindness-detection -p /content/aptos && cd /content/aptos && unzip -q -o "*.zip"')
        meta = pd.read_csv('/content/aptos/train.csv')
        X = np.stack([_to224(Image.open(f'/content/aptos/train_images/{i}.png')) for i in meta.id_code])
        y = meta.diagnosis.to_numpy().astype(int)
    np.savez_compressed(CACHE, X=X, y=y)
    return X, y

X_ALL, Y_ALL = load_aptos()
if MAX_IMAGES and len(Y_ALL) > MAX_IMAGES:
    keep = np.sort(np.random.default_rng(0).choice(len(Y_ALL), MAX_IMAGES, replace=False))
    X_ALL, Y_ALL = X_ALL[keep], Y_ALL[keep]
print('images', X_ALL.shape, '| grade counts', np.bincount(Y_ALL, minlength=5))

# near-duplicate groups by perceptual hash (union-find over pairs within DUP_HAMMING bits)
hashes = [imagehash.phash(Image.fromarray(x)) for x in X_ALL]
H = np.array([np.asarray(h.hash, dtype=bool).ravel() for h in hashes])
parent = list(range(len(H)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
for i in range(len(H)):
    d = (H[i + 1:] != H[i]).sum(1)
    for j in np.where(d <= DUP_HAMMING)[0] + i + 1:
        parent[find(j)] = find(i)
GROUPS = np.array([find(i) for i in range(len(H))])
sizes = pd.Series(GROUPS).value_counts()
dup_groups = int((sizes > 1).sum()); dup_images = int(sizes[sizes > 1].sum())
conflict = int(sum(len(set(Y_ALL[GROUPS == g])) > 1 for g in sizes[sizes > 1].index))
print(f'near-duplicate groups: {dup_groups} covering {dup_images} images; groups with conflicting grades: {conflict}')
json.dump({'dup_groups': dup_groups, 'dup_images': dup_images, 'conflicting_label_groups': conflict},
          open(os.path.join(OUT_DIR, f'duplicates_{RUN_TAG}.json'), 'w'))

## Step 4 — Model, training, MC dropout, Grad-CAM and text reports
EfficientNet-B0 (ImageNet weights) with a dropout head. MC dropout keeps only the head's dropout active, so the
30 stochastic passes reuse one feature extraction per image.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
print('device', DEV, '| torch', torch.__version__)

def split(seed):
    """Test 20 %, validation ~10 %, training ~70 %; near-duplicates always stay together."""
    idx = np.arange(len(Y_ALL))
    tr_va, te = next(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(idx, Y_ALL, GROUPS))
    tr_i, va_i = next(StratifiedGroupKFold(8, shuffle=True, random_state=seed).split(tr_va, Y_ALL[tr_va], GROUPS[tr_va]))
    return tr_va[tr_i], tr_va[va_i], te

def to_tensor(xb):
    t = torch.from_numpy(xb).permute(0, 3, 1, 2).float() / 255.0
    return (t - MEAN) / STD

def build_model(pretrained=True):
    w = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if (pretrained and not SYNTHETIC_TEST) else None
    m = torchvision.models.efficientnet_b0(weights=w)
    m.classifier = nn.Sequential(nn.Dropout(DROPOUT_P), nn.Linear(m.classifier[1].in_features, 5))
    return m.to(DEV)

def augment(t):
    if np.random.rand() < .5: t = torch.flip(t, [3])
    if np.random.rand() < .5: t = torch.flip(t, [2])
    k = np.random.randint(4); t = torch.rot90(t, k, [2, 3])
    return t * (1 + 0.1 * (torch.rand(t.size(0), 1, 1, 1) - .5))

def train(seed, tr, va):
    torch.manual_seed(seed); np.random.seed(seed)
    m = build_model(); opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(EPOCHS, 1))
    best, best_state = -1, None
    for ep in range(EPOCHS):
        m.train(); perm = np.random.permutation(tr)
        for b in range(0, len(perm), 32):
            ib = perm[b:b + 32]
            xb = augment(to_tensor(X_ALL[ib])).to(DEV); yb = torch.from_numpy(Y_ALL[ib]).to(DEV)
            loss = F.cross_entropy(m(xb), yb, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        p = predict_det(m, va).argmax(1); k = cohen_kappa_score(Y_ALL[va], p, weights='quadratic')
        if k > best:
            best, best_state = k, {kk: v.detach().cpu().clone() for kk, v in m.state_dict().items()}
        print(f'  seed {seed} epoch {ep+1}/{EPOCHS} val QWK={k:.3f}')
    m.load_state_dict(best_state); m.eval()
    return m, best

@torch.no_grad()
def features(m, idx, bs=64):
    m.eval(); out = []
    for b in range(0, len(idx), bs):
        f = m.avgpool(m.features(to_tensor(X_ALL[idx[b:b + bs]]).to(DEV))).flatten(1); out.append(f)
    return torch.cat(out)

@torch.no_grad()
def predict_det(m, idx):
    return F.softmax(m.classifier[1](features(m, idx)), 1).cpu().numpy()

@torch.no_grad()
def predict_mc(m, idx, T=MC_SAMPLES):
    f = features(m, idx); lin = m.classifier[1]
    P = torch.stack([F.softmax(lin(F.dropout(f, DROPOUT_P, training=True)), 1) for _ in range(T)])
    return P.mean(0).cpu().numpy()

def entropy(p):
    return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1)

def gradcam(m, xb, cls):
    """Grad-CAM on the last convolutional block; returns 7x7 maps and 224x224 maps in [0,1]."""
    m.eval(); store = {}
    layer = m.features[-1]
    h = layer.register_forward_hook(lambda mod, i, o: store.__setitem__('a', o))
    xb = xb.to(DEV)
    with torch.enable_grad():
        out = m(xb); a = store['a']
        g = torch.autograd.grad(out[torch.arange(len(cls)), torch.as_tensor(cls, device=DEV)].sum(), a)[0]
    h.remove()
    cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1)).detach()       # (B, 7, 7)
    cam = cam / (cam.flatten(1).max(1)[0].view(-1, 1, 1) + 1e-8)
    up = F.interpolate(cam[:, None], size=(IMG, IMG), mode='bilinear', align_corners=False)[:, 0]
    return cam.cpu().numpy(), up.cpu().numpy()

BLUR = torchvision.transforms.GaussianBlur(31, sigma=10)
@torch.no_grad()
def conf_with_mask(m, xb, mask, cls):
    """Replace masked pixels by a blurred copy (not black) and return confidence of class cls."""
    xb = xb.to(DEV); blurred = BLUR(xb); mk = torch.from_numpy(mask).float().to(DEV)[:, None]
    x2 = xb * (1 - mk) + blurred * mk
    return F.softmax(m(x2), 1)[torch.arange(len(cls)), torch.as_tensor(cls, device=DEV)].cpu().numpy()

def cell_mask(cells_idx, grid):
    """Boolean 224x224 mask from selected cells of a grid x grid partition."""
    s = IMG / grid; M = np.zeros((IMG, IMG), np.float32)
    for c in cells_idx:
        r, q = divmod(int(c), grid); M[int(r * s):int((r + 1) * s), int(q * s):int((q + 1) * s)] = 1
    return M

REGION = [['upper-left', 'upper-centre', 'upper-right'], ['middle-left', 'central', 'middle-right'],
          ['lower-left', 'lower-centre', 'lower-right']]

def text_report(grade, conf, unc, thr, region, area):
    flag = unc > thr
    return (f'Predicted grade: {GRADES[grade]} (confidence {conf:.0%}). '
            f'The model relied mostly on the {region} region of the image; '
            f'{area:.0%} of the image received strong attention. '
            f'Uncertainty is {"HIGH" if flag else "low"} (entropy {unc:.2f} vs threshold {thr:.2f}): '
            + ('refer to an ophthalmologist for review.' if flag else 'automated grade can be reported.'))

## Step 5 — Run the study for each seed (resumable)
For every seed: train, predict (deterministic and MC dropout), compute Grad-CAM, faithfulness and stability tests,
and the text-faithfulness test on the test images. Per-image results are saved to Drive.

In [ ]:
PER_IMAGE_CSV = os.path.join(OUT_DIR, f'per_image_{RUN_TAG}.csv')
SEED_CSV = os.path.join(OUT_DIR, f'per_seed_{RUN_TAG}.csv')
done = set(pd.read_csv(SEED_CSV)['seed']) if os.path.exists(SEED_CSV) else set()
REPORTS = []

for seed in SEEDS:
    if seed in done:
        print('seed', seed, 'already done'); continue
    t0 = time.time(); tr, va, te = split(seed)
    assert not set(GROUPS[te]) & set(GROUPS[tr]), 'duplicate leakage between train and test'
    m, val_qwk = train(seed, tr, va)
    pd_te, pm_te = predict_det(m, te), predict_mc(m, te)
    pd_va, pm_va = predict_det(m, va), predict_mc(m, va)
    u_mc_te, u_mc_va = entropy(pm_te), entropy(pm_va)
    u_sm_te, u_sm_va = 1 - pd_te.max(1), 1 - pd_va.max(1)
    pred = pm_te.argmax(1); conf = pm_te.max(1)
    thr_main = float(np.quantile(u_mc_va, 1 - MAIN_REFERRAL))

    rows = []; rng = np.random.default_rng(seed)
    for b in range(0, len(te), 32):
        ib = te[b:b + 32]; xb = to_tensor(X_ALL[ib]); cls = pred[b:b + 32]
        cam7, cam224 = gradcam(m, xb, cls)
        # stability: CAM under a small, label-preserving perturbation
        xb_noisy = xb + 0.03 * torch.randn_like(xb)
        cam7_n, _ = gradcam(m, xb_noisy, cls)
        base = conf_with_mask(m, xb, np.zeros((len(ib), IMG, IMG), np.float32), cls)
        # visual faithfulness on a 7x7 grid: top cells vs random cells covering MASK_FRACTION of the image
        k = max(1, int(round(MASK_FRACTION * 49)))
        top = np.stack([cell_mask(np.argsort(-c.ravel())[:k], 7) for c in cam7])
        rnd = np.stack([cell_mask(rng.choice(49, k, replace=False), 7) for _ in range(len(ib))])
        c_top, c_rnd = conf_with_mask(m, xb, top, cls), conf_with_mask(m, xb, rnd, cls)
        # textual faithfulness on a 3x3 grid: the region named in the text vs another random region
        cam3 = np.stack([np.array([c[int(r*IMG/3):int((r+1)*IMG/3), int(q*IMG/3):int((q+1)*IMG/3)].mean()
                                   for r in range(3) for q in range(3)]) for c in cam224])
        named = cam3.argmax(1); other = np.array([rng.choice([j for j in range(9) if j != n]) for n in named])
        c_named = conf_with_mask(m, xb, np.stack([cell_mask([n], 3) for n in named]), cls)
        c_other = conf_with_mask(m, xb, np.stack([cell_mask([o], 3) for o in other]), cls)
        for j, i in enumerate(ib):
            gi = b + j
            stab = pd.Series(cam7[j].ravel()).corr(pd.Series(cam7_n[j].ravel()), method='spearman')
            region = REGION[named[j] // 3][named[j] % 3]; area = float((cam224[j] > 0.5).mean())
            rows.append({'seed': seed, 'img': int(i), 'y': int(Y_ALL[i]), 'pred_mc': int(pred[gi]), 'pred_det': int(pd_te[gi].argmax()),
                         'conf_mc': float(conf[gi]), 'unc_mc_entropy': float(u_mc_te[gi]), 'unc_softmax': float(u_sm_te[gi]),
                         'p_referable_mc': float(pm_te[gi, 2:].sum()), 'flag_main': bool(u_mc_te[gi] > thr_main),
                         'conf_base': float(base[j]), 'drop_top': float(base[j] - c_top[j]), 'drop_random': float(base[j] - c_rnd[j]),
                         'faithfulness_gap': float(c_rnd[j] - c_top[j]), 'stability_spearman': float(stab),
                         'text_region': region, 'drop_named_region': float(base[j] - c_named[j]),
                         'drop_other_region': float(base[j] - c_other[j]), 'text_faithful': bool(c_named[j] < c_other[j]),
                         'attention_area': area})
            if len(REPORTS) < 40:
                REPORTS.append({'seed': seed, 'img': int(i), 'true': GRADES[int(Y_ALL[i])],
                                'report': text_report(int(pred[gi]), float(conf[gi]), float(u_mc_te[gi]), thr_main, region, area)})
    df_img = pd.DataFrame(rows)
    df_img.to_csv(PER_IMAGE_CSV, mode='a', index=False, header=not os.path.exists(PER_IMAGE_CSV))

    # per-seed summary incl. selective prediction on the test set (thresholds fixed on validation)
    def qwk(a, b): return cohen_kappa_score(a, b, weights='quadratic') if len(set(a)) > 1 else np.nan
    yt = Y_ALL[te]; err = (pred != yt)
    srow = {'seed': seed, 'n_train': len(tr), 'n_val': len(va), 'n_test': len(te), 'val_qwk': val_qwk,
            'acc': accuracy_score(yt, pred), 'macro_f1': f1_score(yt, pred, average='macro'), 'qwk': qwk(yt, pred),
            'auc_referable': roc_auc_score(yt >= 2, pm_te[:, 2:].sum(1)) if len(set(yt >= 2)) > 1 else np.nan,
            'minutes': (time.time() - t0) / 60}
    for name, u_te, u_va in [('mc', u_mc_te, u_mc_va), ('softmax', u_sm_te, u_sm_va)]:
        srow[f'err_auroc_{name}'] = roc_auc_score(err, u_te) if 0 < err.sum() < len(err) else np.nan
        for r in REFERRAL_RATES:
            thr = np.quantile(u_va, 1 - r) if r > 0 else np.inf
            keep = u_te <= thr
            srow[f'{name}_r{int(r*100)}_referred'] = 1 - keep.mean()
            srow[f'{name}_r{int(r*100)}_qwk_kept'] = qwk(yt[keep], pred[keep])
            srow[f'{name}_r{int(r*100)}_acc_kept'] = accuracy_score(yt[keep], pred[keep]) if keep.any() else np.nan
            srow[f'{name}_r{int(r*100)}_errors_caught'] = (err & ~keep).sum() / max(err.sum(), 1)
    # calibration (expected calibration error, 15 bins) for MC and deterministic predictions
    for name, P in [('mc', pm_te), ('det', pd_te)]:
        c, p_ = P.max(1), P.argmax(1); bins = np.linspace(0, 1, 16); e = 0
        for lo, hi in zip(bins[:-1], bins[1:]):
            s = (c > lo) & (c <= hi)
            if s.any(): e += s.mean() * abs((p_[s] == yt[s]).mean() - c[s].mean())
        srow[f'ece_{name}'] = e
    pd.DataFrame([srow]).to_csv(SEED_CSV, mode='a', index=False, header=not os.path.exists(SEED_CSV))
    torch.save(m.state_dict(), os.path.join(OUT_DIR, f'model_seed{seed}_{RUN_TAG}.pt'))
    print(f'seed {seed}: QWK={srow["qwk"]:.3f} acc={srow["acc"]:.3f} | errors caught at 20%: '
          f'MC={srow["mc_r20_errors_caught"]:.2f} softmax={srow["softmax_r20_errors_caught"]:.2f} | {srow["minutes"]:.1f} min')
if REPORTS:
    pd.DataFrame(REPORTS).to_csv(os.path.join(OUT_DIR, f'example_text_reports_{RUN_TAG}.csv'), index=False)

## Step 6 — Analysis tables (saved to Drive)
- **Table A:** grading performance and calibration (mean ± std over seeds)
- **Table B (RQ1):** selective prediction — referral rate, QWK/accuracy of retained cases, share of errors caught, for MC dropout vs softmax
- **Table C (RQ2):** visual explanation faithfulness and stability, flagged vs unflagged cases
- **Table D (RQ3):** text-explanation faithfulness rate, flagged vs unflagged

In [ ]:
from scipy.stats import mannwhitneyu, binomtest, wilcoxon
S = pd.read_csv(SEED_CSV); D = pd.read_csv(PER_IMAGE_CSV)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

tabA = S[['acc', 'macro_f1', 'qwk', 'auc_referable', 'ece_mc', 'ece_det', 'err_auroc_mc', 'err_auroc_softmax']].agg(['mean', 'std']).T.round(4)

rowsB = []
for name in ['mc', 'softmax']:
    for r in REFERRAL_RATES:
        p = f'{name}_r{int(r*100)}'
        rowsB.append({'uncertainty': 'MC dropout entropy' if name == 'mc' else 'Softmax (1 - max prob)', 'target_referral': r,
                      'actual_referred': S[f'{p}_referred'].mean(), 'qwk_retained': S[f'{p}_qwk_kept'].mean(),
                      'qwk_retained_std': S[f'{p}_qwk_kept'].std(), 'acc_retained': S[f'{p}_acc_kept'].mean(),
                      'errors_caught': S[f'{p}_errors_caught'].mean()})
tabB = pd.DataFrame(rowsB).round(4)
def boot_auroc_diff(D, B=2000, seed=0):
    """Paired bootstrap over test images (within each seed) for AUROC(MC entropy) - AUROC(softmax)."""
    rng = np.random.default_rng(seed); diffs = []
    groups = [g for _, g in D.groupby('seed')]
    for _ in range(B):
        d = []
        for g in groups:
            i = rng.integers(0, len(g), len(g)); e = (g.pred_mc.values != g.y.values)[i]
            if 0 < e.sum() < len(e):
                d.append(roc_auc_score(e, g.unc_mc_entropy.values[i]) - roc_auc_score(e, g.unc_softmax.values[i]))
        if d: diffs.append(np.mean(d))
    diffs = np.asarray(diffs)
    if len(diffs) < 100: return np.nan, np.nan
    return float(diffs.mean()), float(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())))
auroc_diff, p_err = boot_auroc_diff(D)

def compare(col):
    a, b = D.loc[~D.flag_main, col].dropna(), D.loc[D.flag_main, col].dropna()
    p = mannwhitneyu(a, b).pvalue if len(a) > 1 and len(b) > 1 else np.nan
    return {'metric': col, 'unflagged_mean': a.mean(), 'flagged_mean': b.mean(), 'p_mannwhitney': p, 'n_unflagged': len(a), 'n_flagged': len(b)}
tabC = pd.DataFrame([compare('faithfulness_gap'), compare('drop_top'), compare('drop_random'), compare('stability_spearman')]).round(4)
faithful_share = (D.faithfulness_gap > 0).mean()
p_faith = wilcoxon(D.drop_top, D.drop_random).pvalue if len(D) > 10 else np.nan

rowsD = []
for name, sub in [('all', D), ('unflagged', D[~D.flag_main]), ('flagged', D[D.flag_main])]:
    k, n = int(sub.text_faithful.sum()), len(sub)
    rowsD.append({'cases': name, 'n': n, 'text_faithful_rate': k / n if n else np.nan,
                  'p_vs_chance_0.5': binomtest(k, n, 0.5, alternative='greater').pvalue if n else np.nan,
                  'mean_drop_named': sub.drop_named_region.mean(), 'mean_drop_other': sub.drop_other_region.mean()})
tabD = pd.DataFrame(rowsD).round(4)
region_counts = D.text_region.value_counts()

for name, tab in [('tableA_performance', tabA), ('tableB_selective', tabB), ('tableC_visual_xai', tabC), ('tableD_text_xai', tabD)]:
    tab.to_csv(os.path.join(OUT_DIR, f'{name}_{RUN_TAG}.csv'))
json.dump({'share_faithful_visual': faithful_share, 'p_drop_top_vs_random': p_faith, 'auroc_diff_mc_minus_softmax': auroc_diff, 'p_err_auroc_mc_vs_softmax': p_err,
           'text_region_counts': region_counts.to_dict()}, open(os.path.join(OUT_DIR, f'summary_stats_{RUN_TAG}.json'), 'w'), indent=2, default=float)
if len(SEEDS) < 3:
    print('NOTE: pilot (1 seed, 2 epochs): numbers only check that the pipeline runs.\n')
print('TABLE A — performance\n', tabA, '\n\nTABLE B — selective prediction (RQ1)\n', tabB,
      f'\nError-detection AUROC, MC minus softmax: {auroc_diff:+.4f} (paired bootstrap p = {p_err:.4f})',
      '\n\nTABLE C — visual explanations (RQ2)\n', tabC,
      f'\nShare of images where hiding the top Grad-CAM region hurts more than a random region: {faithful_share:.3f} (Wilcoxon p = {p_faith:.3g})',
      '\n\nTABLE D — textual explanations (RQ3)\n', tabD, '\n\nRegions named in the texts:\n', region_counts.to_string())

## Step 7 — Figures (300 dpi PNG and vector PDF, saved to Drive)

In [ ]:
import matplotlib.pyplot as plt
FIG = os.path.join(OUT_DIR, f'figures_{RUN_TAG}'); os.makedirs(FIG, exist_ok=True)
def save(fig, name):
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{FIG}/{name}.{ext}', dpi=300)
    plt.show()

# Fig 1: referral rate vs QWK of retained cases, and errors caught
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for name, col in [('MC dropout entropy', '#2a78b8'), ('Softmax (1 - max prob)', '#d9822b')]:
    s = tabB[tabB.uncertainty == name]
    axes[0].errorbar(100 * s.actual_referred, s.qwk_retained, yerr=s.qwk_retained_std.fillna(0), marker='o', color=col, label=name, capsize=2)
    axes[1].plot(100 * s.actual_referred, 100 * s.errors_caught, marker='o', color=col, label=name)
axes[0].set_xlabel('Cases referred (%)'); axes[0].set_ylabel('QWK of retained cases')
axes[1].set_xlabel('Cases referred (%)'); axes[1].set_ylabel('Errors caught by referral (%)'); axes[1].legend(fontsize=8)
save(fig, 'fig1_referral')

# Fig 2: explanation faithfulness and stability, unflagged vs flagged
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, col, lab in [(axes[0], 'faithfulness_gap', 'Faithfulness gap\n(conf. drop top − random)'),
                     (axes[1], 'stability_spearman', 'Stability (Spearman ρ)'),
                     (axes[2], None, 'Text faithful (%)')]:
    if col:
        ax.boxplot([D.loc[~D.flag_main, col].dropna(), D.loc[D.flag_main, col].dropna()], showfliers=False)
        ax.set_xticks([1, 2]); ax.set_xticklabels(['Unflagged', 'Flagged'])   # works on every Matplotlib version
        if col == 'faithfulness_gap': ax.axhline(0, color='grey', lw=.8, ls='--')
    else:
        v = [100 * D.loc[~D.flag_main, 'text_faithful'].mean(), 100 * D.loc[D.flag_main, 'text_faithful'].mean()]
        ax.bar(['Unflagged', 'Flagged'], v, color=['#2a78b8', '#c4473a']); ax.axhline(50, color='grey', lw=.8, ls='--')
    ax.set_title(lab, fontsize=9)
save(fig, 'fig2_explanation_quality')

# Fig 3: example reports (image, Grad-CAM overlay, text)
seed0 = SEEDS[0]
m0 = build_model(); m0.load_state_dict(torch.load(os.path.join(OUT_DIR, f'model_seed{seed0}_{RUN_TAG}.pt'), map_location=DEV)); m0.eval()
ex = D[D.seed == seed0]
picks = pd.concat([ex[~ex.flag_main & (ex.y == ex.pred_mc)].head(2), ex[ex.flag_main].head(2)]).img.tolist()[:4]
te0 = np.array(picks)
pm = predict_mc(m0, te0); cls = pm.argmax(1); _, cams = gradcam(m0, to_tensor(X_ALL[te0]), cls)
thr0 = float(np.quantile(entropy(predict_mc(m0, split(seed0)[1])), 1 - MAIN_REFERRAL))
fig, axes = plt.subplots(len(picks), 3, figsize=(10, 2.7 * len(picks)), gridspec_kw={'width_ratios': [1, 1, 2]}, squeeze=False)
for r, i in enumerate(picks):
    row = ex[ex.img == i].iloc[0]
    axes[r][0].imshow(X_ALL[i]); axes[r][0].set_title(f'True: {GRADES[int(row.y)]}', fontsize=8)
    axes[r][1].imshow(X_ALL[i]); axes[r][1].imshow(cams[r], cmap='jet', alpha=.45); axes[r][1].set_title('Grad-CAM', fontsize=8)
    txt = text_report(int(cls[r]), float(pm[r].max()), float(entropy(pm[r:r+1])[0]), thr0, row.text_region, row.attention_area)
    axes[r][2].text(0, .5, textwrap.fill(txt, 60), fontsize=7.5, va='center', family='monospace')
    for a in axes[r]: a.axis('off')
save(fig, 'fig3_example_reports')

## Step 8 — Record the environment (for the reproducibility section)

In [ ]:
import sklearn
env = {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
       'scikit-learn': sklearn.__version__, 'device': DEV,
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'data_source': DATA_SOURCE, 'n_images': int(len(Y_ALL)), 'grade_counts': np.bincount(Y_ALL, minlength=5).tolist(),
       'seeds': SEEDS, 'epochs': EPOCHS, 'image_size': IMG, 'dropout_p': DROPOUT_P, 'mc_samples': MC_SAMPLES,
       'main_referral': MAIN_REFERRAL, 'mask_fraction': MASK_FRACTION, 'dup_hamming': DUP_HAMMING,
       **json.load(open(os.path.join(OUT_DIR, f'duplicates_{RUN_TAG}.json')))}
json.dump(env, open(os.path.join(OUT_DIR, f'environment_{RUN_TAG}.json'), 'w'), indent=2)
env